# SUC - Example 1

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Deterministic UC with a fixed solar output (50 MW).

In [1]:
from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, SolverFactory,
    Binary, minimize, value
)

c1, SU1, Pgmin1, Pgmax1 = 10, 800, 40, 80
c2, SU2, Pgmin2, Pgmax2 = 30, 100, 20, 90
TLoad = 120
SolarPg = 50

m = ConcreteModel()
m.u1 = Var(domain=Binary); m.u2 = Var(domain=Binary)
m.Pg1 = Var(); m.Pg2 = Var()

m.obj = Objective(
    expr=(m.u1*SU1 + c1*m.Pg1) + (m.u2*SU2 + c2*m.Pg2), sense=minimize
)
m.PowerBalance   = Constraint(expr=m.Pg1 + m.Pg2 == TLoad - SolarPg)
m.genLimit_1_Min = Constraint(expr=Pgmin1*m.u1 <= m.Pg1)
m.genLimit_1_Max = Constraint(expr=m.Pg1 <= Pgmax1*m.u1)
m.genLimit_2_Min = Constraint(expr=Pgmin2*m.u2 <= m.Pg2)
m.genLimit_2_Max = Constraint(expr=m.Pg2 <= Pgmax2*m.u2)

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
m = model
print(f"u1 = {value(m.u1)}, u2 = {value(m.u2)}")
print(f"Pg1 = {value(m.Pg1):.4f}, Pg2 = {value(m.Pg2):.4f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpdq2_qi8y.pyomo.lp


Reading time = 0.00 seconds
x1: 5 rows, 4 columns, 10 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))



CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads



Non-default parameters:
TimeLimit  90
MIPGap  0

Optimize a model with 5 rows, 4 columns and 10 nonzeros


Model fingerprint: 0x46cc692e
Variable types: 2 continuous, 2 integer (2 binary)
Coefficient statistics:


...

  Bounds range     [1e+00, 1e+00]
  RHS range        [7e+01, 7e+01]


Presolve removed 5 rows and 4 columns


Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 20 available processors)



Solution count 1: 1500 

Optimal solution found (tolerance 0.00e+00)
Best objective 1.500000000000e+03, best bound 1.500000000000e+03, gap 0.0000%


ok optimal
u1 = 1.0, u2 = 0.0
Pg1 = 70.0000, Pg2 = 0.0000
